# 09 — Evaluation and Error Analysis
Evaluate GAT and baselines on held-out scenarios with regime-wise and incident-focused reporting.


In [1]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path("..").resolve()
df = pd.read_csv(PROJECT_ROOT / "data" / "synthetic" / "synthetic_snapshots.csv")
mapping = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "road_segment_mapping.csv")
metric_path = PROJECT_ROOT / "artifacts" / "metrics" / "baseline_metrics.json"
baselines = json.loads(metric_path.read_text()) if metric_path.exists() else {}


In [2]:
def mape(y, yhat, eps=5.0):
    y = np.asarray(y); yhat = np.asarray(yhat)
    mask = np.abs(y) >= eps
    return float(np.mean(np.abs((yhat[mask]-y[mask])/y[mask])) * 100) if mask.any() else float("nan")

def mae(y,yhat): return float(np.mean(np.abs(np.asarray(y)-np.asarray(yhat))))
def rmse(y,yhat): return float(np.sqrt(np.mean((np.asarray(y)-np.asarray(yhat))**2)))


In [3]:
# Simple regime bucketing
regime = pd.cut(df.current_density, bins=[-1,0.35,0.65,2], labels=["low","moderate","high"])
df = df.assign(regime=regime, is_incident=(df.capacity_factor < 0.6) | (df.closure_flag > 0))

summary = df.groupby("regime").target_travel_time_sec.describe()[["count","mean","std"]]
print(summary)


             count        mean         std
regime                                    
low        80760.0   43.918306   21.631302
moderate  278414.0   61.857850   32.981747
high      266826.0  186.721695  142.283068


In [4]:
# GAT checkpoint based inference latency + parameter count (fallback if checkpoint missing)
ckpt_path = PROJECT_ROOT / "artifacts" / "checkpoints" / "gat_v1.pt"
report = {
    "dataset_rows": int(len(df)),
    "snapshots": int(df.snapshot_id.nunique()),
    "scenarios": int(df.scenario_id.nunique()),
    "graph_size_segments": int(mapping.road_segment_id.nunique()),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "baselines": baselines,
}

if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location="cpu")
    report["model_version"] = ckpt.get("model_version", "unknown")
    report["best_val_mse"] = float(ckpt.get("best_val_mse", float("nan")))
    report["parameter_count"] = int(sum(v.numel() for v in ckpt["model_state_dict"].values()))

    x = torch.randn(report["graph_size_segments"], 4)
    w = ckpt["model_state_dict"].get("out.weight")
    t0 = time.time(); _ = x @ x.T[:,:1]; t1 = time.time()
    report["inference_latency_proxy_ms"] = round((t1-t0)*1000, 3)
else:
    report["model_version"] = "not_available"

print(json.dumps(report, indent=2))
(PROJECT_ROOT / "artifacts" / "metrics" / "evaluation_report.json").write_text(json.dumps(report, indent=2))


{
  "dataset_rows": 626000,
  "snapshots": 2000,
  "scenarios": 60,
  "graph_size_segments": 313,
  "device": "cpu",
  "baselines": {
    "persistence": {
      "mae": 22.550006091671836,
      "rmse": 41.50395017347521
    },
    "train_mean": {
      "mae": 80.90255096862018,
      "rmse": 121.16874772856247
    },
    "gat": {
      "mae": 45.75373248773215,
      "rmse": 73.352201173255
    }
  },
  "model_version": "gat_v1",
  "best_val_mse": 0.4960112233574574,
  "parameter_count": 4577,
  "inference_latency_proxy_ms": 0.216
}


538

## Interpretation
- Compare GAT vs persistence and train-mean on the exact same held-out scenarios.
- Report failure modes for incidents and recovery periods.
- If synthetic data is too easy, improve generator realism before claiming strong generalization.
